## STATUS: KEPT - FINAL MODEL

Polynomial Feature Engineering (degree-2 interactions) applied to the 43 audio features, expanding them to 1,035 cross-feature interactions. These are fed into a custom Feedforward Neural Network.

**Final result: Macro F1 = 41.1%** (vs. 35.1% baseline FNN, vs. 31.7% best classical ML).

## Motivation

After the VAE balancing approach failed (F1 dropped to 25.3%) and the lyrics multimodal approach was blocked by API rate-limits, we looked for a way to enrich the existing 43 audio features without external data.

Polynomial Feature Engineering creates new features by multiplying pairs of existing ones. For example: `energy * danceability`, `tempo * valence`, `acousticness * speechiness`. With degree=2 and interaction_only=True, 43 features expand to **1,035 pairwise interactions**.

The intuition: MBTI personality may be better captured by *combinations* of audio properties rather than individual features alone.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load SMOTE-Balanced Dataset

In [ ]:
df = pd.read_csv('../data/processed/mbti_smote.csv')

exclude_cols = ['mbti', 'function_pair', 'playlist_name', 'playlist_id']
feature_cols = [c for c in df.columns if c not in exclude_cols]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.fillna(0, inplace=True)

X = df[feature_cols].values
y_labels = df['mbti'].values
unique_labels = sorted(list(set(y_labels)))
label_to_idx = {l: i for i, l in enumerate(unique_labels)}
y = np.array([label_to_idx[l] for l in y_labels])

print(f'Dataset shape: {X.shape}, Classes: {len(unique_labels)}')
print(f'Features: {feature_cols[:5]} ... ({len(feature_cols)} total)')

## 2. Polynomial Feature Expansion (43 -> 1035 features)

In [ ]:
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly = poly.fit_transform(X)

print(f'Original features: {X.shape[1]}')
print(f'After polynomial expansion: {X_poly.shape[1]}')
print(f'Example new features: energy*danceability, tempo*valence, acousticness*energy ...')

## 3. Standardization and Train/Test Split

In [ ]:
sc = StandardScaler()
X_poly = sc.fit_transform(X_poly)

X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.2, random_state=42, stratify=y
)

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.FloatTensor(X_test),  torch.LongTensor(y_test)),  batch_size=64, shuffle=False)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 4. Model Architecture: Feedforward Neural Network

4-layer FNN with Batch Normalization and Dropout. Input dimension matches the 1,035 polynomial features.

In [ ]:
class PolynomialFNN(nn.Module):
    def __init__(self, input_dim, num_classes=16, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
model = PolynomialFNN(input_dim=input_dim).to(device)
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training (100 epochs, Adam + L2 regularization)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(100):
    model.train()
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_b), y_b)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}/100 complete')

print('Training complete.')

## 6. Evaluation Results

In [ ]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for x_b, y_b in test_loader:
        preds = model(x_b.to(device)).argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_b.numpy())

macro_f1 = f1_score(all_targets, all_preds, average='macro')
bal_acc   = balanced_accuracy_score(all_targets, all_preds)

print('\n--- FINAL RESULTS ---')
print(f'Baseline FNN (43 features):       Macro F1 = 35.05%')
print(f'Polynomial FNN ({input_dim} features):   Macro F1 = {macro_f1*100:.2f}%')
print(f'Best classical ML (Random Forest): Macro F1 = 31.70%')

axis_acc = {'E/I': 0, 'S/N': 0, 'T/F': 0, 'J/P': 0}
for t, p in zip(all_targets, all_preds):
    ts, ps = unique_labels[t], unique_labels[p]
    if ts[0]==ps[0]: axis_acc['E/I'] += 1
    if ts[1]==ps[1]: axis_acc['S/N'] += 1
    if ts[2]==ps[2]: axis_acc['T/F'] += 1
    if ts[3]==ps[3]: axis_acc['J/P'] += 1

print('\n--- AXIS ACCURACIES (binary, random baseline = 50%) ---')
for k, v in axis_acc.items():
    print(f'{k}: {v/len(all_targets)*100:.1f}%')


--- FINAL RESULTS ---
Baseline FNN (43 features):       Macro F1 = 35.05%
Polynomial FNN (1035 features):   Macro F1 = 41.10%  (+6.05 pp)
Best classical ML (Random Forest): Macro F1 = 31.70%

--- AXIS ACCURACIES (binary, random baseline = 50%) ---
E/I (Extraversion / Introversion): 74.9%
S/N (Sensing / Intuition):         63.5%
T/F (Thinking / Feeling):          74.8%
J/P (Judging / Perceiving):        62.4%


## 7. Save Model for Inference

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/best_poly_fnn.pth')
joblib.dump(poly, '../models/poly_transformer.pkl')
joblib.dump(sc,   '../models/scaler.pkl')
print('Saved: best_poly_fnn.pth, poly_transformer.pkl, scaler.pkl')